# CYK Parser Success Rate Evaluation

This notebook evaluates the CYK parser success rate across all California committee hearings in the corpus.

In [1]:
from pathlib import Path
import subprocess
import zipfile

# This is where the unzipped corpus file is stored
CORPUS_FILE_PATH = 'DH2024_Corpus_Release/'
corpus_dir = Path(CORPUS_FILE_PATH)
zip_path = Path("digitaldemocracy-2015-2018/DH2024_Corpus_Release.zip")
repo_dir = Path("digitaldemocracy-2015-2018")

# Clone repository if not present
if not repo_dir.is_dir():
    print("Corpus directory not found. Cloning repository...")
    subprocess.run(
        ["git", "clone", "https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018"],
        check=True,
    )
    print("Repository cloned successfully.")

# Extract zip file if corpus directory doesn't exist
if not corpus_dir.is_dir():
    if zip_path.exists():
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Extraction complete.")
    else:
        print(f"Error: {zip_path} not found!")
else:
    print(f"Corpus already extracted at {corpus_dir}")

Corpus already extracted at DH2024_Corpus_Release


## Load California Hearings

In [2]:
from src import HearingLoader, HearingTagger
from src.grammar.Parser import Parser
import time

# Initialize the HearingLoader with the corpus path
loader = HearingLoader(corpus_path='DH2024_Corpus_Release/')
tagger = HearingTagger()
parser = Parser()

print("Loading all committee hearings...")
start_time = time.time()
all_hearings = loader.load_all_committee_hearings()
load_time = time.time() - start_time
print(f"Total hearings loaded: {len(all_hearings)} (took {load_time:.1f}s)")

# Filter for California hearings only
ca_hearings = [h for h in all_hearings if h.state == 'CA']
print(f"California hearings: {len(ca_hearings)}")

# Filter out hearings with no bill discussed
ca_hearings_with_bills = [h for h in ca_hearings if h.bid != 'CA_NO BILL DISCUSSED']
print(f"California hearings with bills: {len(ca_hearings_with_bills)}")

Loading all committee hearings...
invalid literal for int() with base 10: 'hid'
invalid literal for int() with base 10: 'hid'
invalid literal for int() with base 10: 'hid'
Total hearings loaded: 8218 (took 10.5s)
California hearings: 8218
California hearings with bills: 7069


## Run Tagger and Evaluate CYK Parser (Combined)

To improve efficiency, we'll tag and parse in a single loop with progress tracking.

In [ ]:
import numpy as np

print("Processing hearings (tagging + parsing)...")
print("This will take several minutes...\n")

parse_results = []
successful_parses = []
failed_parses = []
tagging_errors = []

total = len(ca_hearings_with_bills)
start_time = time.time()

for i, hearing in enumerate(ca_hearings_with_bills):
    # Progress update every 50 hearings
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (total - i - 1) / rate if rate > 0 else 0
        print(f"Processed {i + 1}/{total} hearings ({(i+1)/total*100:.1f}%)")
    
    try:
        # Tag the hearing
        tagged_hearing = tagger(hearing)
        
        # Try to parse
        try:
            parse_trees = list(parser.get_all_parses_as_nltk_trees(tagged_hearing, max_parses=1) or [])
            parse_success = len(parse_trees) > 0
            
            parse_results.append(int(parse_success))
            
            if parse_success:
                successful_parses.append((hearing.hid, hearing.bid, len(parse_trees)))
            else:
                failed_parses.append((hearing.hid, hearing.bid))
                
        except Exception as e:
            # Parsing failed with exception
            parse_results.append(0)
            failed_parses.append((hearing.hid, hearing.bid))
            
    except Exception as e:
        # Tagging failed
        tagging_errors.append((hearing.hid, hearing.bid, str(e)))
        continue

total_time = time.time() - start_time
print(f"\nProcessing complete! Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"Successfully processed: {len(parse_results)}/{total} hearings")
print(f"Tagging errors: {len(tagging_errors)}")

Processing hearings (tagging + parsing)...
This will take several minutes...

Processed 100/7069 hearings (1.4%)
Processed 200/7069 hearings (2.8%)
Processed 300/7069 hearings (4.2%)
Processed 400/7069 hearings (5.7%)
Processed 500/7069 hearings (7.1%)
Processed 600/7069 hearings (8.5%)
Processed 700/7069 hearings (9.9%)
Processed 800/7069 hearings (11.3%)
Processed 900/7069 hearings (12.7%)


In [ ]:
tagging_errors

## Results

In [ ]:
total_hearings = len(parse_results)
successful_count = np.sum(parse_results)
failed_count = total_hearings - successful_count
success_rate = np.mean(parse_results) * 100 if parse_results else 0

print("=" * 80)
print("CYK Parser Statistics for 2015-2018 California Hearings")
print("=" * 80)
print(f"\nTotal hearings evaluated: {total_hearings}")
print(f"Successful parses:\t{successful_count}")
print(f"Failed parses:\t{failed_count}")
print(f"\nSuccess rate:\t{success_rate:.2f}%")
print("\n" + "=" * 80)

## Detailed Statistics

In [ ]:
successful_parses[:5]

In [ ]:
print("\nSuccessful Parses Statistics:")
print(f"  Total: {len(successful_parses)}")

if successful_parses:
    parse_tree_counts = [count for _, _, count in successful_parses]
    print(f"  Average parse trees per hearing: {np.mean(parse_tree_counts):.2f}")
    print(f"  Min parse trees: {np.min(parse_tree_counts)}")
    print(f"  Max parse trees: {np.max(parse_tree_counts)}")
    
    # Show distribution of parse tree counts
    unique, counts = np.unique(parse_tree_counts, return_counts=True)
    print("\n  Parse tree count distribution:")
    for tree_count, freq in zip(unique, counts):
        print(f"    {tree_count} tree(s): {freq} hearings ({freq/len(successful_parses)*100:.1f}%)")

print(f"\nFailed Parses:")
print(f"  Total: {len(failed_parses)}")

# Show first 10 failed parses as examples
if failed_parses:
    print("\n  Examples of failed parses (first 10):")
    for i, (hid, bid) in enumerate(failed_parses[:10]):
        print(f"    {i+1}. Hearing ID: {hid}, Bill: {bid}")

if tagging_errors:
    print(f"\nTagging Errors:")
    print(f"  Total: {len(tagging_errors)}")
    print("\n  Examples (first 5):")
    for i, (hid, bid, error) in enumerate(tagging_errors[:5]):
        print(f"    {i+1}. Hearing ID: {hid}, Bill: {bid}")
        print(f"       Error: {error[:100]}..." if len(error) > 100 else f"       Error: {error}")